# Laboratorio L11 - Sovradattamento e generalizzazione

B.6 - Data science e Machine Learning: dai dati ai modelli

File necessari: `tragitti_puliti.csv` (oppure il file pulito della classe, prodotto in L5), `pinguini.csv`.

Dati: Palmer Penguins, Horst AM, Hill AP, Gorman KB (2020), licenza CC0, https://allisonhorst.github.io/palmerpenguins/

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def controlla(condizione, suggerimento="Non ancora: rileggere la consegna."):
    """Stampa un riscontro immediato sull'esercizio."""
    print("Corretto." if condizione else suggerimento)

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

tragitti = pd.read_csv("tragitti_puliti.csv").dropna(subset=["distanza_km", "tempo_min"])
X = tragitti[["distanza_km", "tempo_min"]]
y = tragitti["mezzo"]
X_add, X_ver, y_add, y_ver = train_test_split(X, y, test_size=0.3, random_state=1, stratify=y)
print("addestramento:", len(X_add), " verifica:", len(X_ver))
print(y.value_counts())

Problema: prevedere il mezzo di trasporto dalla distanza e dal tempo. È più difficile dei pinguini: mezzi diversi si sovrappongono (piedi, bici e monopattino per distanze brevi; bus e auto per distanze medie).

## 1. Curva addestramento-verifica (esercizio 1)

Per ogni profondità massima dell'albero si calcola l'accuratezza sui due insiemi.

In [ ]:
profondita = list(range(1, 13))
acc_add, acc_ver = [], []
for p in profondita:
    albero = DecisionTreeClassifier(max_depth=p, random_state=0).fit(X_add, y_add)
    acc_add.append(albero.score(X_add, y_add))
    acc_ver.append(albero.score(X_ver, y_ver))

plt.plot(profondita, acc_add, marker="o", label="addestramento")
plt.plot(profondita, acc_ver, marker="s", label="verifica")
plt.xlabel("profondità massima")
plt.ylabel("accuratezza")
plt.legend()
plt.show()
pd.DataFrame({"profondità": profondita, "addestramento": acc_add, "verifica": acc_ver}).round(3)

**Domande**

- Da quale profondità l'accuratezza di addestramento continua a crescere mentre quella di verifica smette di migliorare?
- Dove si trova il sottoadattamento? Dove il sovradattamento?

Risposte:

Completare: trovare la profondità con l'accuratezza di verifica più alta. `np.argmax(lista)` restituisce la posizione del valore massimo.

In [ ]:
migliore = profondita[np.___(acc_ver)]
print("profondità migliore sulla verifica:", migliore)
controlla(migliore == 5)

## 2. Il risultato dipende dalla suddivisione (esercizio 2)

Si ripete la valutazione dell'albero di profondità 5 con 10 suddivisioni casuali diverse (semi da 0 a 9).

In [ ]:
risultati = []
for seme in range(10):
    a, b, c, d = train_test_split(X, y, test_size=0.3, random_state=seme, stratify=y)
    risultati.append(DecisionTreeClassifier(max_depth=5, random_state=0).fit(a, c).score(b, d))
print(np.round(risultati, 3))
print("minimo", round(min(risultati), 3), " massimo", round(max(risultati), 3), " media", round(np.mean(risultati), 3))

**Domanda**: se due modelli differiscono di 0,02 di accuratezza su una sola suddivisione, si può dire che uno è migliore dell'altro?

Risposta:

## 3. Validazione incrociata (esercizio 3)

`cross_val_score(modello, X, y, cv=5)`: divide i dati in 5 blocchi; per 5 volte addestra su 4 blocchi e valuta sul quinto, cambiando ogni volta il blocco di verifica. Restituisce le 5 accuratezze. La media è una stima più stabile di una singola suddivisione.

In [ ]:
punteggi = cross_val_score(DecisionTreeClassifier(max_depth=5, random_state=0), X, y, cv=5)
print(np.round(punteggi, 3), " media:", round(punteggi.mean(), 3))

Scegliere la profondità con la validazione incrociata. La tabella riporta media e deviazione standard delle 5 accuratezze.

In [ ]:
righe = []
for p in [1, 2, 3, 4, 5, 6, 8, None]:
    s = cross_val_score(DecisionTreeClassifier(max_depth=p, random_state=0), X, y, cv=5)
    righe.append({"profondità": str(p), "media": s.mean(), "dev. std.": s.std()})
pd.DataFrame(righe).round(3)

**Domanda**: quale profondità scegliereste? Se due profondità hanno medie quasi uguali, quale conviene e perché?

Risposta:

## 4. k-NN: scegliere k (esercizio 4)

Completare il ciclo: per ogni k calcolare la media della validazione incrociata di un k-NN. Le due caratteristiche (km e minuti) hanno scale diverse ma confrontabili; per semplicità qui non si standardizza.

In [ ]:
for k in [1, 3, 5, 9, 15, 25]:
    s = cross_val_score(KNeighborsClassifier(n_neighbors=___), X, y, cv=5)
    print(f"k = {k:2}: media {s.mean():.3f}")

## 5. La verifica finale si usa una volta sola (esercizio 5)

Procedura corretta:

1. scegliere modello e iperparametri con la validazione incrociata sui soli dati di addestramento
2. addestrare il modello scelto su tutti i dati di addestramento
3. misurare una sola volta l'accuratezza sull'insieme di verifica, tenuto da parte fin dall'inizio

Completare: scegliere la profondità con la validazione incrociata su `X_add`, `y_add`, poi valutare sulla verifica.

In [ ]:
medie = {}
for p in [2, 3, 4, 5, 6, 8]:
    medie[p] = cross_val_score(DecisionTreeClassifier(max_depth=p, random_state=0), X_add, y_add, cv=5).mean()
scelta = max(medie, key=medie.get)          # la profondità con la media più alta
print({p: round(v, 3) for p, v in medie.items()}, "-> scelta:", scelta)

finale = DecisionTreeClassifier(max_depth=scelta, random_state=0).fit(X_add, y_add)
acc_finale = finale.score(___, ___)
print("accuratezza sulla verifica:", round(acc_finale, 3))
controlla(0.5 < acc_finale < 0.9)

**Domanda**: perché non si deve scegliere la profondità guardando direttamente l'accuratezza sulla verifica, come nell'esercizio 1?

Risposta: